# Week 2 · Day 2 — Transfer Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kumarsaurabh1817/Final-Computer-Vision-Training/blob/main/week2_classification/day2_transfer_learning/Week2_Day2_Colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-repo-181717?logo=github)](https://github.com/kumarsaurabh1817/Final-Computer-Vision-Training)

**Feature Extraction vs Full Fine-Tuning**

| | |
|---|---|
| **Runtime** | GPU (T4 — free tier) |
| **Estimated time** | ~60–90 min (10 epochs each mode) |
| **Topics** | Freeze / unfreeze backbone · Two-LR strategy · Replace classifier head · Compare strategies side-by-side |

---
### ⚡ Before you start
Go to **Runtime → Change runtime type → T4 GPU** then run cells top-to-bottom.

In [ ]:
# ── 0a · Clone course repo from GitHub ─────────────────────────────────
import os, sys

REPO_URL = 'https://github.com/kumarsaurabh1817/Final-Computer-Vision-Training.git'
REPO_DIR = '/content/Final-Computer-Vision-Training'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    print('Repo already present — pulling latest...')
    !git -C $REPO_DIR pull

sys.path.insert(0, REPO_DIR)
print(f'Repo ready at: {REPO_DIR}')


## 0 · Setup

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

In [ ]:
!pip install -q wandb
print('✅ Packages ready')

In [ ]:
# ── 0b · Google Drive mount ─────────────────────────────────────────────
# Set True to persist results & model checkpoints across Colab sessions
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    _BASE       = '/content/drive/MyDrive/ComputerVision/week2_classification'
    RESULTS_DIR = _BASE + '/results'
    MODEL_DIR   = _BASE + '/models'
else:
    RESULTS_DIR = '/content/results'
    MODEL_DIR   = '/content/models'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,   exist_ok=True)
print(f'Results -> {RESULTS_DIR}')
print(f'Models  -> {MODEL_DIR}')


## 1 · Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.models as models
from torchvision.models import ResNet18_Weights, ResNet50_Weights, EfficientNet_B0_Weights
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import numpy as np
import time
import json

torch.manual_seed(42)
np.random.seed(42)
print('✅ Imports done')

## 2 · Training Configuration
Adjust these to trade off speed vs accuracy.

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────────
BACKBONE    = 'resnet18'     # 'resnet18' | 'resnet50' | 'efficientnet_b0'
EPOCHS      = 10             # 5 for a quick test, 20 for a serious run
BATCH_SIZE  = 64
LR_HEAD     = 1e-3           # learning rate for the new classifier head
LR_BACKBONE = 1e-5           # learning rate for the backbone (fine-tuning only)
WEIGHT_DECAY= 1e-4
IMG_SIZE    = 300 if 'b3' in BACKBONE else 224
NUM_CLASSES = 10             # CIFAR-10
USE_WANDB   = False          # set True and run wandb.login() cell below
# ──────────────────────────────────────────────────────────────

print(f'Backbone  : {BACKBONE}')
print(f'Epochs    : {EPOCHS}')
print(f'Img size  : {IMG_SIZE}px')
print(f'LR head / backbone : {LR_HEAD} / {LR_BACKBONE}')

In [ ]:
# Optional W&B login — skip if USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.login()   # paste your API key from wandb.ai/settings
    print('✅ Logged into W&B')

## 3 · Concept Explainer

In [ ]:
print("""
TRANSFER LEARNING — TWO STRATEGIES
====================================

Strategy A — Feature Extraction (freeze backbone):
  • Freeze ALL backbone weights (requires_grad = False)
  • Only train the new head (normal LR: 1e-3)
  • Fast training, few params updated
  • Use when: small dataset OR domain is similar to ImageNet

Strategy B — Full Fine-Tuning (unfreeze backbone):
  • Allow ALL weights to update
  • Use TWO different LRs:
      - backbone : 1e-5  (tiny — don't destroy pretrained features)
      - head     : 1e-3  (normal — learn from scratch)
  • Slower but higher accuracy ceiling
  • Use when: larger dataset OR domain differs from ImageNet

⚠️ Risk of fine-tuning at high backbone LR:
  'Catastrophic forgetting' — the rich ImageNet features get
  overwritten by gradients from your small task.
  → Always use discriminative LRs.
""")

## 4 · Data — CIFAR-10 with ImageNet Normalisation

In [ ]:
# ImageNet normalisation stats — required for pretrained backbones!
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

transform_train = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomCrop(IMG_SIZE, padding=8),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_eval = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = torchvision.datasets.CIFAR10('/content/data', train=True,  download=True, transform=transform_train)
test_set   = torchvision.datasets.CIFAR10('/content/data', train=False, download=True, transform=transform_eval)

val_size   = int(len(full_train) * 0.1)
train_size = len(full_train) - val_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                   generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {train_size:,}  Val: {val_size:,}  Test: {len(test_set):,}')
print(f'Image size: {IMG_SIZE}×{IMG_SIZE}  (resized from 32×32)')

## 5 · Model Builder — Replace Head, Freeze/Unfreeze

In [ ]:
SUPPORTED = {
    'resnet18':       (models.resnet18,       ResNet18_Weights.DEFAULT),
    'resnet50':       (models.resnet50,       ResNet50_Weights.DEFAULT),
    'efficientnet_b0':(models.efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
}

def build_model(backbone_name, num_classes, freeze_backbone):
    """
    Load pretrained backbone → replace head → optionally freeze backbone.
    """
    builder, weights = SUPPORTED[backbone_name]
    model = builder(weights=weights)

    # Step 1: freeze everything
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False

    # Step 2: replace classifier head
    if 'resnet' in backbone_name:
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        )
    elif 'efficientnet' in backbone_name:
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        )

    # Step 3: re-enable gradients on the new head
    head = model.fc if 'resnet' in backbone_name else model.classifier[-1]
    for p in head.parameters():
        p.requires_grad = True

    return model.to(DEVICE)


def param_summary(model, label):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[{label}] Total: {total:,}  Trainable: {trainable:,}  ({100*trainable/total:.1f}%)')


# Preview both modes
print('--- Feature Extraction ---')
m = build_model(BACKBONE, NUM_CLASSES, freeze_backbone=True)
param_summary(m, 'Feature Extraction')
del m

print('\n--- Full Fine-Tuning ---')
m = build_model(BACKBONE, NUM_CLASSES, freeze_backbone=False)
param_summary(m, 'Full Fine-Tuning')
del m

## 6 · Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()
    return loss_sum / total, 100 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()
    return loss_sum / total, 100 * correct / total

print('✅ Training functions defined')

In [ ]:
def run_training(freeze_backbone, label, wandb_run=None):
    print(f'\n{"─"*55}')
    print(f'  {label}  ({BACKBONE.upper()}, {EPOCHS} epochs)')
    print(f'{"─"*55}')

    model     = build_model(BACKBONE, NUM_CLASSES, freeze_backbone)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    if freeze_backbone:
        # Only head params — single LR
        opt_params = [p for p in model.parameters() if p.requires_grad]
        optimizer  = optim.AdamW(opt_params, lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    else:
        # Two-LR: backbone + head
        head = model.fc if 'resnet' in BACKBONE else model.classifier[-1]
        head_ids = {id(p) for p in head.parameters()}
        backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
        optimizer = optim.AdamW([
            {'params': backbone_params, 'lr': LR_BACKBONE},
            {'params': list(head.parameters()), 'lr': LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_val = 0.0
    mode_key = 'feat_ext' if freeze_backbone else 'finetune'

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        tl, ta = train_epoch(model, train_loader, criterion, optimizer)
        vl, va = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tl)
        history['train_acc'].append(ta)
        history['val_loss'].append(vl)
        history['val_acc'].append(va)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Ep {epoch:2d}/{EPOCHS}  lr={lr_now:.2e}  '
              f'Train {tl:.3f}/{ta:.1f}%  Val {vl:.3f}/{va:.1f}%')

        if wandb_run:
            wandb_run.log({f'{mode_key}/epoch':epoch, f'{mode_key}/val_acc':va,
                           f'{mode_key}/train_acc':ta, f'{mode_key}/lr':lr_now})

        if va > best_val:
            best_val = va
            torch.save(model.state_dict(), f'{MODEL_DIR}/week2_{BACKBONE}_{mode_key}.pt')

    elapsed = time.time() - t0
    model.load_state_dict(torch.load(f'{MODEL_DIR}/week2_{BACKBONE}_{mode_key}.pt', weights_only=True, map_location=DEVICE))
    _, test_acc = evaluate(model, test_loader, criterion)

    print(f'\n  Best val acc : {best_val:.1f}%')
    print(f'  Test acc     : {test_acc:.1f}%')
    print(f'  Time         : {elapsed:.0f}s')

    history['best_val'] = best_val
    history['test_acc'] = test_acc
    return history

print('✅ run_training defined')

## 7 · Strategy A — Feature Extraction
(backbone frozen, only new head trains)

In [ ]:
wandb_run = None
if USE_WANDB:
    import wandb
    wandb_run = wandb.init(project='cv-week2-transfer-learning',
                           name=f'{BACKBONE}-comparison',
                           config={'backbone':BACKBONE,'epochs':EPOCHS,
                                   'lr_head':LR_HEAD,'lr_backbone':LR_BACKBONE})

feat_history = run_training(freeze_backbone=True,
                             label='Strategy A — Feature Extraction',
                             wandb_run=wandb_run)

## 8 · Strategy B — Full Fine-Tuning
(backbone + head, two learning rates)

In [ ]:
ft_history = run_training(freeze_backbone=False,
                           label='Strategy B — Full Fine-Tuning',
                           wandb_run=wandb_run)

if USE_WANDB and wandb_run:
    wandb_run.finish()

## 9 · Compare Results — Plots

In [ ]:
epochs = range(1, len(feat_history['train_acc']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Transfer Learning Comparison — {BACKBONE.upper()}', fontsize=13)

for i, (metric, ylabel) in enumerate([('acc', 'Accuracy (%)'), ('loss', 'Loss')]):
    ax = axes[i]
    ax.plot(epochs, feat_history[f'train_{metric}'], '--', color='#4C72B0', label='FeatExt Train')
    ax.plot(epochs, feat_history[f'val_{metric}'],         color='#4C72B0', label='FeatExt Val', lw=2)
    ax.plot(epochs, ft_history[f'train_{metric}'],   '--', color='#C44E52', label='FineTune Train')
    ax.plot(epochs, ft_history[f'val_{metric}'],           color='#C44E52', label='FineTune Val', lw=2)
    ax.set_title(f'{ylabel} — Train vs Val')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/day2_{BACKBONE}_comparison.png', dpi=120)
plt.show()
print(f'Saved → {RESULTS_DIR}/day2_{BACKBONE}_comparison.png')

## 10 · Results Summary

In [ ]:
print('\n' + '='*50)
print('RESULTS SUMMARY')
print('='*50)
print('  {:<25} {:>9} {:>9}'.format('Strategy', 'Val Acc', 'Test Acc'))
print('  ' + '-'*45)
print('  {:<25} {:>8.1f}% {:>8.1f}%'.format('Feature Extraction', feat_history['best_val'], feat_history['test_acc']))
print('  {:<25} {:>8.1f}% {:>8.1f}%'.format('Full Fine-Tuning', ft_history['best_val'], ft_history['test_acc']))

winner = 'Full Fine-Tuning' if ft_history['test_acc'] > feat_history['test_acc'] else 'Feature Extraction'
diff   = abs(ft_history['test_acc'] - feat_history['test_acc'])
print(f'\n  Winner : ✅ {winner}  (by {diff:.1f}%)')

print("""
Interpretation:
  • If fine-tuning wins: the domain shift from ImageNet→CIFAR-10
    was large enough that the backbone needed to adapt.
  • If feature extraction wins (or ties): ImageNet features already
    capture what CIFAR-10 needs; simpler approach is sufficient.
  • More epochs and a custom dataset will usually widen the gap.
""")

# Save JSON
results = {
    'backbone': BACKBONE,
    'epochs':   EPOCHS,
    'feature_extraction': {'best_val': feat_history['best_val'], 'test_acc': feat_history['test_acc']},
    'full_finetune':      {'best_val': ft_history['best_val'],   'test_acc': ft_history['test_acc']},
}
results_path = f'{RESULTS_DIR}/day2_{BACKBONE}_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results JSON → {results_path}')

## 11 · Download Results to Local Machine

In [ ]:
# Downloads the comparison plot and results JSON to your local Downloads folder
from google.colab import files

files.download(f'{RESULTS_DIR}/day2_{BACKBONE}_comparison.png')
files.download(results_path)
print('✅ Files sent to your browser Downloads folder')

## 12 · Experiment — What Happens with High Backbone LR?
Run this cell to see catastrophic forgetting in action.

In [ ]:
# ⚠️ EXPERIMENT: both groups use the same high LR — watch accuracy collapse
WRONG_LR_BACKBONE = LR_HEAD   # deliberately wrong: same LR for backbone and head

print(f'Running 3-epoch experiment: backbone LR = {WRONG_LR_BACKBONE} (too high!)')

model_bad = build_model(BACKBONE, NUM_CLASSES, freeze_backbone=False)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
head_bad = model_bad.fc if 'resnet' in BACKBONE else model_bad.classifier[-1]
optimizer_bad = optim.AdamW([
    {'params': [p for p in model_bad.parameters() if id(p) not in {id(q) for q in head_bad.parameters()}],
     'lr': WRONG_LR_BACKBONE},          # ← THIS IS WRONG
    {'params': list(head_bad.parameters()), 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

for ep in range(1, 4):
    tl, ta = train_epoch(model_bad, train_loader, criterion, optimizer_bad)
    vl, va = evaluate(model_bad, val_loader, criterion)
    print(f'  Ep {ep}  Train {ta:.1f}%  Val {va:.1f}%  ← compare to fine-tune val above')

print('\n→ If val acc is much lower than the two-LR fine-tune, the high backbone LR hurt!')
del model_bad

---
## ✅ Day 2 Complete!

**What you built:**
- `build_model()` — loads pretrained backbone, replaces head, freezes/unfreezes
- Side-by-side comparison: feature extraction vs fine-tuning with two-LR
- Demonstrated catastrophic forgetting (high backbone LR experiment)

**Next → Day 3:** Pick a Kaggle dataset and build a custom `Dataset` + `DataLoader` pipeline.